# Flood Prediction – Rio Grande do Sul

Objetivo:
Prever a ocorrência de enchentes (nível do rio acima da cota de inundação)
com horizonte de previsão de 3 dias.

Fontes:
- INMET
- ANA
- Atlas Digital de Desastres
- ENSO / Niño Index

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')

## 1. Configurações Globais

In [ ]:
LOOKBACK = 14
FORECAST = 3

RANDOM_STATE = 42

FEATURES = [
    "precipitacao",
    "temperatura",
    "pressao",
    "umidade",
    "vento",
    "nivel_rio",
    "nino_index",
]

TARGET = "enchente_3d"

# Quando os dados estiverem carregados, use:
# df = create_flood_target(df)
# df = create_forecast_target(df, forecast=FORECAST)

## Dataset Schema

Este projeto trabalha com observações por estação hidrológica.

Chave primária:

- data_hora
- estacao_id

Variáveis meteorológicas:
- precipitacao
- temperatura
- pressao
- umidade
- vento

Variáveis hidrológicas:
- nivel_rio
- cota_inundacao

Variáveis climáticas:
- nino_index

Target:
- enchente
- enchente_3d (horizonte de 3 dias)

### Controle de experimentos

In [ ]:
EXPERIMENT_NAME = "baseline_v1"

TRAIN_END = "2022-12-31"
VALID_END = "2023-12-31"

USE_NINO = True
USE_RIVER_LEVEL = True
USE_PRECIP_ACCUM = True

## 2. Carregar e tratar os dados

Os dados foram obtidos a partir de arquivos CSV disponibilizados pelo Instituto Nacional de Meteorologia (INMET), sendo referentes ao ano de 2024 para possibilitar uma análise detalhada de todos os fatores que influenciaram os eventos meteorológicos extremos daquele ano. Fora isso, foram renomeados os atributos, para facilitar sua visualização, além da data e hora que foram convertidas em datetime.

In [ ]:
def load_csv_data(path, date_col="data", sep=";", decimal=",", dayfirst=True):
    df = pd.read_csv(path, sep=sep, decimal=decimal)
    if date_col in df.columns:
        df = df.rename(columns={date_col: "data_hora"})
    df["data_hora"] = pd.to_datetime(df["data_hora"], dayfirst=dayfirst, errors="coerce")
    return df


def load_inmet_data(path):
    return load_csv_data(path, date_col="data")


def load_ana_data(path):
    return load_csv_data(path, date_col="data")


def load_disaster_data(path):
    return load_csv_data(path, date_col="data")


def load_nino_data(path):
    return load_csv_data(path, date_col="data")


# Example placeholders for your local dataset paths.
# Atualize estes caminhos para os arquivos reais antes de executar.
INMET_PATH = "path/to/inmet_data.csv"
ANA_PATH = "path/to/ana_data.csv"
DISASTER_PATH = "path/to/disaster_data.csv"
NINO_PATH = "path/to/nino_index.csv"

# Exemplo de uso:
# weather_df = load_inmet_data(INMET_PATH)
# river_df = load_ana_data(ANA_PATH)
# disaster_df = load_disaster_data(DISASTER_PATH)
# nino_df = load_nino_data(NINO_PATH)

## 3. Integração das fontes

In [ ]:
def merge_datasets(
    weather_df,
    river_df,
    disaster_df,
    nino_df,
    on_keys=("data_hora", "estacao_id"),
):
    df = weather_df.copy()
    df = df.rename(columns={"data": "data_hora"})
    df["data_hora"] = pd.to_datetime(df["data_hora"], errors="coerce")

    if "estacao_id" not in df.columns:
        raise ValueError("weather_df deve conter a coluna 'estacao_id'.")

    river = river_df.copy()
    river = river.rename(columns={"data": "data_hora"})
    river["data_hora"] = pd.to_datetime(river["data_hora"], errors="coerce")

    disaster = disaster_df.copy()
    disaster = disaster.rename(columns={"data": "data_hora"})
    disaster["data_hora"] = pd.to_datetime(disaster["data_hora"], errors="coerce")

    nino = nino_df.copy()
    nino = nino.rename(columns={"data": "data_hora"})
    nino["data_hora"] = pd.to_datetime(nino["data_hora"], errors="coerce")

    df = df.merge(
        river,
        on=["data_hora", "estacao_id"],
        how="left",
        suffixes=("", "_river"),
    )

    df = df.merge(
        disaster,
        on=["data_hora", "estacao_id"],
        how="left",
        suffixes=("", "_disaster"),
    )

    if "nino_index" in nino.columns:
        df = df.merge(nino[["data_hora", "nino_index"]], on="data_hora", how="left")
    else:
        df["nino_index"] = np.nan

    df = df.sort_values(["estacao_id", "data_hora"]).reset_index(drop=True)
    return df


def validate_dataset(df, name="dataset"):
    print(f"==== Validação: {name} ====")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Types:\n", df.dtypes)
    print("Missing values:\n", df.isna().sum())
    print("Unique stations:", df["estacao_id"].nunique() if "estacao_id" in df.columns else "N/A")
    print("Date range:", df["data_hora"].min(), df["data_hora"].max())
    print("===========================\n")
    return df


# Exemplo de uso:
# merged_df = merge_datasets(weather_df, river_df, disaster_df, nino_df)
# validate_dataset(merged_df, name="Merged data")

## 4. Identify target and predictors

In [ ]:
def create_flood_target(df):
    if "nivel_rio" not in df.columns or "cota_inundacao" not in df.columns:
        raise KeyError("As colunas 'nivel_rio' e 'cota_inundacao' são necessárias para criar o target.")

    df = df.copy()
    df["enchente"] = (df["nivel_rio"] >= df["cota_inundacao"]).astype(int)
    return df


def create_forecast_target(df, forecast=FORECAST):
    df = df.sort_values(["estacao_id", "data_hora"])
    df = df.copy()

    def future_max(group):
        return (
            group["enchente"]
            .rolling(window=forecast, min_periods=1)
            .max()
            .shift(-forecast + 1)
        )

    df["enchente_3d"] = df.groupby("estacao_id").apply(future_max).reset_index(level=0, drop=True).fillna(0).astype(int)
    return df


# Exemplo de uso:
# df = create_flood_target(df)
# df = create_forecast_target(df, forecast=FORECAST)
# TARGET = "enchente_3d"  # use para previsão em horizonte de 3 dias

## 3. Feature Engineering

In [ ]:
def create_features(df):
    df = df.sort_values(["estacao_id", "data_hora"]).copy()
    group = df.groupby("estacao_id")

    df["precip_3d"] = group["precipitacao"].rolling(3, min_periods=1).sum().reset_index(level=0, drop=True)
    df["precip_7d"] = group["precipitacao"].rolling(7, min_periods=1).sum().reset_index(level=0, drop=True)
    df["precip_14d"] = group["precipitacao"].rolling(14, min_periods=1).sum().reset_index(level=0, drop=True)

    df["delta_rio_1d"] = group["nivel_rio"].diff(1)
    df["delta_rio_3d"] = group["nivel_rio"].diff(3)

    df["mes"] = df["data_hora"].dt.month
    df["dia_ano"] = df["data_hora"].dt.dayofyear
    df["semana_ano"] = df["data_hora"].dt.isocalendar().week.astype(int)
    df["dia_semana"] = df["data_hora"].dt.weekday
    df["trimestre"] = df["data_hora"].dt.quarter

    return df


# Exemplo de uso:
# df = create_features(df)
# validate_dataset(df, name="With features")

## 4. Criação das sêquencias temporais

In [ ]:
feature_cols = [
    "precipitacao",
    "temperatura",
    "pressao",
    "umidade",
    "vento",
    "nivel_rio",
    "nino_index",
]

numeric_features = feature_cols + [
    "precip_3d",
    "precip_7d",
    "precip_14d",
    "delta_rio_1d",
    "delta_rio_3d",
]


def build_tabular_dataset(df, feature_columns, target_column):
    X = df[feature_columns].copy()
    y = df[target_column].copy()
    return X, y


def create_sequences(df, feature_columns, lookback, forecast):
    X, y = [], []
    features = df[feature_columns].values
    target = df[TARGET].values

    for i in range(len(df) - lookback - forecast + 1):
        X.append(features[i : i + lookback])
        y.append(target[i + lookback : i + lookback + forecast])

    return np.array(X), np.array(y)


# Para o modelo LSTM, normalizamos apenas os recursos usados pelo modelo sequencial.
sequence_df = df.sort_values(["estacao_id", "data_hora"]).reset_index(drop=True)
sequence_df[numeric_features] = MinMaxScaler().fit_transform(sequence_df[numeric_features].fillna(0))

X_seq, y_seq = create_sequences(sequence_df, feature_cols, LOOKBACK, FORECAST)

# Para modelos tabulares você pode usar:
# X_tab, y_tab = build_tabular_dataset(df, numeric_features, TARGET)
# X_train, X_valid, y_train, y_valid = train_test_split(X_tab, y_tab, test_size=0.2, shuffle=False)


KeyboardInterrupt: 

## 5. Divisão Temporal


In [ ]:
train_df = df[df["data_hora"] <= TRAIN_END].copy()
valid_df = df[(df["data_hora"] > TRAIN_END) & (df["data_hora"] <= VALID_END)].copy()
test_df = df[df["data_hora"] > VALID_END].copy()

print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test:", test_df.shape)

# Exemplo de uso com dados tabulares:
# X_train, y_train = build_tabular_dataset(train_df, numeric_features, TARGET)
# X_valid, y_valid = build_tabular_dataset(valid_df, numeric_features, TARGET)
# X_test, y_test = build_tabular_dataset(test_df, numeric_features, TARGET)

corr = df.corr(numeric_only=True)
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Matriz de correlação")
plt.show()

corr

(4868, 39) (1217, 39)


## 6. Baseline model

In [ ]:
categorical_features = ["mes", "dia_semana", "semana_ano", "trimestre"]
numeric_features_model = [
    "precipitacao",
    "temperatura",
    "pressao",
    "umidade",
    "vento",
    "nivel_rio",
    "nino_index",
    "precip_3d",
    "precip_7d",
    "precip_14d",
    "delta_rio_1d",
    "delta_rio_3d",
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features_model),
        ("cat", categorical_transformer, categorical_features),
    ]
)

X_train, y_train = build_tabular_dataset(train_df, numeric_features_model + categorical_features, TARGET)
X_valid, y_valid = build_tabular_dataset(valid_df, numeric_features_model + categorical_features, TARGET)

baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

baseline_model.fit(X_train, y_train)
pred_valid_baseline = baseline_model.predict(X_valid)

results = []

acc_baseline = accuracy_score(y_valid, pred_valid_baseline)
recall_baseline = recall_score(y_valid, pred_valid_baseline, average='macro')
f1_baseline = f1_score(y_valid, pred_valid_baseline, average='macro')

print("Baseline Accuracy:", acc_baseline)
print("Baseline Recall:", recall_baseline)
print("Baseline F1-Score:", f1_baseline)

results.append({
    "model": "Baseline", 
    "accuracy": acc_baseline,
    "recall": recall_baseline,
    "f1_score": f1_baseline,
})

random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)),
])

random_forest_model.fit(X_train, y_train)
pred_valid_random_forest = random_forest_model.predict(X_valid)

acc_random_forest = accuracy_score(y_valid, pred_valid_random_forest)
recall_random_forest = recall_score(y_valid, pred_valid_random_forest, average='macro')
f1_random_forest = f1_score(y_valid, pred_valid_random_forest, average='macro')

print("Random Forest Accuracy:", acc_random_forest)
print("Random Forest Recall:", recall_random_forest)
print("Random Forest F1-Score:", f1_random_forest)

results.append({
    "model": "Random Forest",
    "accuracy": acc_random_forest,
    "recall": recall_random_forest,
    "f1_score": f1_random_forest,
})

xgboost_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb.XGBClassifier(n_estimators=100, random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='logloss')),
])

xgboost_model.fit(X_train, y_train)
pred_valid_xgboost = xgboost_model.predict(X_valid)

acc_xgboost = accuracy_score(y_valid, pred_valid_xgboost)
recall_xgboost = recall_score(y_valid, pred_valid_xgboost, average='macro')
f1_xgboost = f1_score(y_valid, pred_valid_xgboost, average='macro')

print("XGBoost Accuracy:", acc_xgboost)
print("XGBoost Recall:", recall_xgboost)
print("XGBoost F1-Score:", f1_xgboost)

results.append({
    "model": "XGBoost",
    "accuracy": acc_xgboost,
    "recall": recall_xgboost,
    "f1_score": f1_xgboost,
})

lightgbm_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", lgb.LGBMClassifier(n_estimators=100, random_state=RANDOM_STATE)),
])

lightgbm_model.fit(X_train, y_train)
pred_valid_lightgbm = lightgbm_model.predict(X_valid)

acc_lightgbm = accuracy_score(y_valid, pred_valid_lightgbm)
recall_lightgbm = recall_score(y_valid, pred_valid_lightgbm, average='macro')
f1_lightgbm = f1_score(y_valid, pred_valid_lightgbm, average='macro')

print("LightGBM Accuracy:", acc_lightgbm)
print("LightGBM Recall:", recall_lightgbm)
print("LightGBM F1-Score:", f1_lightgbm)

results.append({
    "model": "LightGBM",
    "accuracy": acc_lightgbm,
    "recall": recall_lightgbm,
    "f1_score": f1_lightgbm,
})

results_df = pd.DataFrame(results)
results_df

Baseline Accuracy: 0.7937551355792933
Baseline F1-Score: 0.7924090289741316


c:\Users\pedro\Desktop\6mestre\ds\lab06_files\lab06_files\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 7. Improved model

In [ ]:
# Ajuste o número de características conforme a lista feature_cols
improved_model = Sequential([
    LSTM(64, activation='tanh', return_sequences=True, input_shape=(LOOKBACK, len(feature_cols))),
    Dropout(0.2),
    LSTM(32, activation='tanh', return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(FORECAST, activation='sigmoid'),   # 3-day forecast, one probability per day
])

improved_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

# Use uma divisão temporal para as sequências.
split_idx = int(len(X_seq) * 0.85)
X_train_seq, X_valid_seq = X_seq[:split_idx], X_seq[split_idx:]
y_train_seq, y_valid_seq = y_seq[:split_idx], y_seq[split_idx:]

improved_model.fit(
    X_train_seq,
    y_train_seq,
    epochs=20,
    batch_size=32,
    validation_data=(X_valid_seq, y_valid_seq),
    verbose=1,
)

pred_valid_improved = (improved_model.predict(X_valid_seq) > 0.5).astype(int)

# Avaliação usando o primeiro dia do horizonte para uma comparação direta
acc_improved = accuracy_score(y_valid_seq[:, 0], pred_valid_improved[:, 0])
recall_improved = recall_score(y_valid_seq[:, 0], pred_valid_improved[:, 0], average='macro')
f1_improved = f1_score(y_valid_seq[:, 0], pred_valid_improved[:, 0], average='macro')

print("Improved Accuracy: ", acc_improved)
print("Improved Recall: ", recall_improved)
print("Improved F1-Score:", f1_improved)

results.append({
    "model": "Improved LSTM",
    "accuracy": acc_improved,
    "recall": recall_improved,
    "f1_score": f1_improved,
})

[LightGBM] [Info] Number of positive: 2439, number of negative: 2429
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002220 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3070
[LightGBM] [Info] Number of data points in the train set: 4868, number of used features: 110
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.501027 -> initscore=0.004108
[LightGBM] [Info] Start training from score 0.004108
Improved Accuracy:  0.8151191454396056
Improved F1-Score: 0.8148771628087155


c:\Users\pedro\Desktop\6mestre\ds\lab06_files\lab06_files\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 8. Compare models

Briefly discuss the difference between the baseline and the improved model.


In [ ]:
print("Distribuição de classes (enchente):")
print(df["enchente"].value_counts(normalize=True))

# O modelo LSTM prevê um horizonte de 3 dias.
probs = improved_model.predict(X_valid_seq)
preds = (probs > 0.5).astype(int)

print("=== Day +1 Forecast ===")
print(classification_report(y_valid_seq[:, 0], preds[:, 0], target_names=['No Flood', 'Flood']))
print(confusion_matrix(y_valid_seq[:, 0], preds[:, 0]))

print("=== Day +2 Forecast ===")
print(classification_report(y_valid_seq[:, 1], preds[:, 1], target_names=['No Flood', 'Flood']))
print(confusion_matrix(y_valid_seq[:, 1], preds[:, 1]))

print("=== Day +3 Forecast ===")
print(classification_report(y_valid_seq[:, 2], preds[:, 2], target_names=['No Flood', 'Flood']))
print(confusion_matrix(y_valid_seq[:, 2], preds[:, 2]))

comparison = pd.DataFrame(results)
comparison

,Model,Accuracy,F1-Score
0,Baseline,0.793755,0.792409
1,Improved,0.815119,0.814877


## 9. Interpretabilidade
primeira coisa a ser descartada caso falte tempo


In [ ]:
# Interpretação de modelos de árvore usando SHAP
preprocessed_valid = preprocessor.transform(X_valid)
feature_names = preprocessor.get_feature_names_out()

explainer = shap.TreeExplainer(xgboost_model.named_steps['model'])
shap_values = explainer.shap_values(preprocessed_valid)

shap.summary_plot(
    shap_values,
    preprocessed_valid,
    feature_names=feature_names,
    show=False,
)
plt.tight_layout()
plt.show()

[LightGBM] [Info] Number of positive: 3065, number of negative: 3020
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002072 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3091
[LightGBM] [Info] Number of data points in the train set: 6085, number of used features: 111
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503698 -> initscore=0.014791
[LightGBM] [Info] Start training from score 0.014791
Submission shape: (2608, 2)
Any missing predictions: False
Any non-binary predictions: False


c:\Users\pedro\Desktop\6mestre\ds\lab06_files\lab06_files\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,PassengerId,prediction
0,8137_01,False
1,7191_01,False
2,6837_01,True
3,4453_01,True
4,4400_03,True


## 10. Gerar predição atual


In [ ]:
# Exemplo de uso para gerar predição com o último bloco de dados disponíveis.
# Substitua '123' pelo estacao_id que deseja prever, ou use a última estação do conjunto.
station_id = df_seq["estacao_id"].iloc[-1]
recent_14_days = (
    df_seq[df_seq["estacao_id"] == station_id]
    .sort_values("data_hora")
    .tail(LOOKBACK)[feature_cols]
    .values
)

if recent_14_days.shape[0] != LOOKBACK:
    raise ValueError(f"Não há dados suficientes para criar a sequência de {LOOKBACK} dias para a estação {station_id}.")

input_tensor = recent_14_days.reshape(1, LOOKBACK, len(feature_cols))
forecast = improved_model.predict(input_tensor)[0]

for i, prob in enumerate(forecast, start=1):
    print(f"Day +{i} flood probability: {prob:.2%}")

# O forecast representa a probabilidade de enchente para os próximos 3 dias.

Saved submission.csv


## Research Hypotheses

H1:
Acúmulo de chuva em 7 dias possui maior poder preditivo
do que chuva diária.

H2:
Nível do rio é a variável mais importante.

H3:
Eventos El Niño aumentam a frequência de enchentes.

H4:
Modelos baseados em árvores superam regressão logística.

H5:
LSTM supera modelos tabulares quando há dados suficientes.